In [15]:
import pandas as pd
import re
import unicodedata

In [215]:
from google.colab import files

uploaded = files.upload()

Saving P10-audio-orderB-annotated.xlsx to P10-audio-orderB-annotated.xlsx


In [216]:
df = pd.read_excel("P10-audio-orderB-annotated.xlsx")

In [217]:
# Cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize("NFC", text)
    text = text.replace("’", "'")
    text = text.replace("\u200b", "")

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [218]:
def remove_punctuation(text):
    if not isinstance(text, str):
        return ""

    # remove everything except letters, numbers, whitespace, and apostrophe
    text = re.sub(r"[^\wÀ-ÿ\s']", " ", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [219]:
# Filler removal function

FILLER_PATTERN = r"""
\b(
um+|uh+|erm+|er+|hmm+|mm+|mhm+|uhm+|ah+|eh+|hm+|huh+|oh+|

okay|ok|yeah|yes|yep|yup|nah|nope|alright|gotcha|

well|so|anyway|basically|actually|literally|
honestly|frankly|obviously|apparently|essentially|

next\ question|
can\ you\ repeat|
can\ you\ explain|
let's\ move|
let's\ see|
let\ me\ see|
do\ you\ mean|
you\ see\ what\ i\ mean|
for\ example|
i\ mean|
you\ know|

kind\ of|
sort\ of|
a\ bit|
relatively|
somewhat|
not\ really|

really|

what\ i\ mean|
how\ to\ say|
let\ me\ think|
wait|
sorry|

thank\ you|

like
)\b
"""


def remove_fillers(text):

    if not isinstance(text, str):
        return ""

    # Remove fillers
    text = re.sub(
        FILLER_PATTERN,
        "",
        text,
        flags=re.IGNORECASE | re.VERBOSE
    )

    # Remove extra spaces created
    text = re.sub(r"\s+", " ", text)

    # Clean punctuation spacing again
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    return text.strip()


In [220]:
def remove_repetition(text):
    if not isinstance(text, str):
        return ""

    # normalize punctuation into spaces
    text = re.sub(r"[,\-;:]", " ", text)

    # collapse repeated words globally (not only adjacent)
    words = text.split()
    cleaned = []

    for i, w in enumerate(words):
        if i == 0 or w.lower() != words[i-1].lower():
            cleaned.append(w)

    return " ".join(cleaned)

In [221]:
# CREATE NEW COLUMNS

# Clean normalized text
df["clean_text"] = df["text"].apply(clean_text)

# Version without punctuation
df["no_punctuation_text"] = df["clean_text"].apply(remove_punctuation)

# Version without fillers
df["no_fillers_text"] = df["no_punctuation_text"].apply(remove_fillers)

# Version without repetitions
df["no_repetition_text"] = df["no_fillers_text"].apply(remove_repetition)

df = df.reindex(columns=[
    "speaker_name",
    "start_time",
    "end_time",
    "text",
    "clean_text",
    "no_punctuation_text",
    "no_fillers_text",
    "no_repetition_text",
    "condition",
    "previous_question",
    "question_type"
])

df = df[df["no_repetition_text"].str.strip() != ""]

In [222]:
# SAVE CLEAN FILE

output_name = "transcript_clean.xlsx"

df.to_excel(output_name, index=False)

print("Cleaning completed.")

# Download cleaned file
files.download(output_name)

Cleaning completed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>